# 06 · Euler, Heun, RK4, pêndulo e Chua

**Objetivo pedagógico:** Estimar ordem por refinamento e separar estabilidade numérica de comportamento físico.

**Luigi Scofano de Araujo · USP 13680334**

Reelaboração de estudo de um tema presente no acervo recuperado de 2025. Esta edição foi preparada em 2026 com assistência de IA. Não é uma cópia autenticada de uma entrega nem uma afirmação de conclusão das questões originais. Os arquivos recuperados estão em `acervo/`, com créditos e limitações em `docs/PROCEDENCIA.md`.

Execute as células em ordem, em kernel novo. Os dados são locais/sintéticos e as conclusões numéricas são verificadas por asserções; gráficos sozinhos não certificam convergência.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd()/"numericos").is_dir() else Path.cwd().parent
if not (ROOT/"numericos").is_dir():
    raise RuntimeError("Execute a partir da raiz do repositório ou de notebooks/.")
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=7, suppress=True)
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True})
FIGURAS = ROOT/"assets"/"figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)
def mostrar(nome):
    plt.tight_layout()
    plt.savefig(FIGURAS/(nome+".svg"), bbox_inches="tight", metadata={"Date": None})
    plt.show()
    plt.close("all")

## 1. Problema de valor inicial
Para u′=u, u(0)=1, a solução exata é exp(t). Euler é de ordem global 1, Heun de ordem 2 e RK4 de ordem 4, sob hipóteses de regularidade e passo adequado. A razão de erros ao reduzir h pela metade tende a 2ᵖ antes do domínio do arredondamento.

In [2]:
from numericos.edo import integrar,circuito_chua,corrente_chua
for method in ["euler","heun","rk4"]:
    errors=[]
    for n in [10,20,40,80]:
        t,u=integrar(lambda t,u:u,(0,1),1.,n,metodo=method)
        errors.append(abs(u[-1]-np.e))
    print(method,"erros:",np.array(errors),"razões:",np.array(errors[:-1])/errors[1:])
    plt.loglog(1/np.array([10,20,40,80]),errors,"o-",label=method)
plt.xlabel("h");plt.ylabel("erro final");plt.legend()
mostrar("06_ordens")

euler erros: [0.1245394 0.0649841 0.033218  0.0167969] razões: [1.9164584 1.9562931 1.9776277]
heun erros: [0.004201  0.0010908 0.0002779 0.0000701] razões: [3.8513766 3.9252845 3.9625631]
rk4 erros: [0.0000021 0.0000001 0.        0.       ] razões: [15.3481759 15.6704045 15.8343018]


## 2. Estabilidade de Euler
Para u′=−20u, Euler explícito tem fator 1−20h. A condição |1−20h|<1 fornece 0<h<0.1. A solução exata decai; um crescimento oscilatório para passo grande é artefato numérico.

In [3]:
for n in [8,40]:
    t,u=integrar(lambda t,u:-20*u,(0,1),1.,n,metodo="euler")
    plt.plot(t,u,"o-",label=f"Euler h={1/n:g}")
t=np.linspace(0,1,300);plt.plot(t,np.exp(-20*t),label="exata")
plt.xlabel("t");plt.ylabel("u");plt.legend()
mostrar("06_estabilidade")

## 3. Pêndulo não linear
θ′=ω e ω′=−(g/L)senθ. A energia específica E=ω²/2+(g/L)(1−cosθ) é constante no modelo sem atrito. RK4 não é um método simplético: erro pequeno numa janela curta não garante conservação por tempos arbitrariamente longos.

In [4]:
g,L=9.81,1.2
f=lambda t,u:np.array([u[1],-g/L*np.sin(u[0])])
t,u=integrar(f,(0,10),[np.pi/2,0],2000)
energy=u[:,1]**2/2+g/L*(1-np.cos(u[:,0]))
print("variação máxima de energia específica:",np.max(abs(energy-energy[0])))
assert np.max(abs(energy-energy[0]))<1e-5
plt.plot(u[:,0],u[:,1]);plt.xlabel("θ (rad)");plt.ylabel("ω (rad/s)")
mostrar("06_pendulo")

variação máxima de energia específica: 2.3513404556751993e-09


## 4. Circuito de Chua em unidades SI
Os parâmetros C₁=10 nF, C₂=100 nF, L=18 mH e a curva por trechos g(V) vêm do material recuperado. O estado é [V₁,V₂,I]. O modelo usa:

$C_1 V_1'=(V_2-V_1)/R-g(V_1)$,
$C_2 V_2'=(V_1-V_2)/R+I$,
$L I'=-V_2$.

As implementações e comparações a seguir foram acrescentadas nesta edição. Não se apresentam como respostas históricas das questões em branco.

In [5]:
from scipy.integrate import solve_ivp
resistance=1800;end=0.008;steps=32000
f=circuito_chua(resistance)
t,u=integrar(f,(0,end),[-0.5,-0.2,0.],steps)
ref=solve_ivp(f,(0,end),[-0.5,-0.2,0.],t_eval=t,method="DOP853",rtol=1e-10,atol=1e-12)
assert ref.success
error=np.max(abs(u-ref.y.T),axis=0)
print("diferença máxima RK4/referência [V,V,A]:",error)
assert error[0]<0.02 and error[1]<0.02 and error[2]<1e-5
fig=plt.figure(figsize=(8,5));ax=fig.add_subplot(111,projection="3d")
ax.plot(u[:,0],u[:,1],1e3*u[:,2],lw=0.7)
ax.set_xlabel("V₁ (V)");ax.set_ylabel("V₂ (V)");ax.set_zlabel("I (mA)")
mostrar("06_chua")

diferença máxima RK4/referência [V,V,A]: [0.0034132 0.0014545 0.000004 ]


## 5. Sensibilidade a R e refinamento
Compare uma janela curta antes de fazer inferências sobre bifurcações. Em um regime caótico, duas aproximações corretas podem divergir ponto a ponto em uma janela longa. Uma imagem de “asas” não basta para certificar um atrator caótico ou um valor crítico de R.

In [6]:
for resistance in [1200,1800,2400]:
    f=circuito_chua(resistance)
    t,coarse=integrar(f,(0,0.002),[-0.5,-0.2,0.],2000)
    _,fine=integrar(f,(0,0.002),[-0.5,-0.2,0.],4000)
    print("R=",resistance,"diferença final por componente:",abs(coarse[-1]-fine[-1]))
    plt.plot(t,coarse[:,0],label=f"R={resistance} Ω")
plt.xlabel("tempo (s)");plt.ylabel("V₁ (V)");plt.legend()
mostrar("06_resistencias")

R= 1200 diferença final por componente: [0.000051  0.0000222 0.0000001]
R= 1800 diferença final por componente: [0.000015  0.0000033 0.       ]
R= 2400 diferença final por componente: [0.        0.0000001 0.       ]


## Para estudar
Implemente um gráfico de erro versus h e estime a inclinação. Discuta a janela de observação e o transiente antes de classificar órbitas. O modelo omite tolerâncias reais dos componentes e não foi validado em bancada.

## Figuras desta execução

![06_chua](../assets/figuras/06_chua.svg)

![06_estabilidade](../assets/figuras/06_estabilidade.svg)

![06_ordens](../assets/figuras/06_ordens.svg)

![06_pendulo](../assets/figuras/06_pendulo.svg)

![06_resistencias](../assets/figuras/06_resistencias.svg)